In [ ]:
!pip uninstall -y bitsandbytes transformers accelerate peft
!pip install -U bitsandbytes transformers accelerate peft datasets evaluate rouge_score meteor

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Found existing installation: transformers 5.4.0
Uninstalling transformers-5.4.0:
  Successfully uninstalled transformers-5.4.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached transformers-5.4.0-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
INFO: pip is looking at multiple versions of meteor to determine which version i

In [ ]:
import os
import torch

!cp /usr/local/cuda/lib64/libcudart.so* /usr/lib/

if torch.cuda.is_available():
    print(f"✅ GPU Bulundu: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Versiyonu: {torch.version.cuda}")
else:
    print("❌ HATA: GPU hala aktif değil!")

✅ GPU Bulundu: Tesla T4
CUDA Versiyonu: 12.6


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False
print("🚀 Model başarıyla yüklendi!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🚀 Model başarıyla yüklendi!


In [ ]:
import json, random, re, torch
from pathlib import Path
from tqdm import tqdm
from transformers import pipeline

chunks_path = Path("/content/chunks (3).jsonl")
output_path = Path("train_ready_qa.jsonl")

rows = []
with chunks_path.open("r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        if "text" in row and len(row["text"].strip()) > 200:
            rows.append(row)

print("Toplam uygun chunk sayısı:", len(rows))

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

Toplam uygun chunk sayısı: 16943


In [ ]:
def generate_qa_data_english_expert(context, page):
    prompt = f"""
You are a world-class Endodontist.
Read the given text and produce exactly one question-answer pair.

Rules:
- The question must be answerable only from the given text.
- The answer must be short, clear, and technical.
- Keep important medical/endodontic keywords.
- Output format must be exactly:

Question: ...
Answer: ...

Text:
{context}
"""

    outputs = generator(
        prompt,
        max_new_tokens=180,
        do_sample=True,
        temperature=0.5,
        top_p=0.9,
        repetition_penalty=1.15,
        return_full_text=False
    )

    raw_text = outputs[0]["generated_text"].strip()

    q_match = re.search(r"Question:\s*(.*)", raw_text)
    a_match = re.search(r"Answer:\s*(.*)", raw_text, re.DOTALL)

    if not q_match or not a_match:
        return None

    question = q_match.group(1).strip()
    answer = a_match.group(1).strip()

    if len(question) < 10 or len(answer) < 15:
        return None

    return {
        "instruction": "Answer as an endodontist using the given context.",
        "input": f"CONTEXT:\n{context}\n\nQuestion: {question}",
        "output": answer
    }

In [ ]:
print("🚀 QA veri seti üretiliyor...")
qa_dataset = []

sample_count = min(500, len(rows))

for ch in tqdm(random.sample(rows, sample_count)):
    context_text = ch["text"][:1000]
    page_num = ch.get("page", "?")

    qa_item = generate_qa_data_english_expert(context_text, page_num)
    if qa_item is not None:
        qa_dataset.append(qa_item)

with output_path.open("w", encoding="utf-8") as f:
    for entry in qa_dataset:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"\n✅ Veri seti başarıyla oluşturuldu.")
print(f"Toplam oluşan örnek sayısı: {len(qa_dataset)}")
print(f"Dosya: '{output_path}'")

🚀 QA veri seti üretiliyor...


  2%|▏         | 9/500 [02:13<1:47:23, 13.12s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 500/500 [1:21:23<00:00,  9.77s/it]


✅ Veri seti başarıyla oluşturuldu.
Toplam oluşan örnek sayısı: 74
Dosya: 'train_ready_qa.jsonl'


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [ ]:
def format_train(ex):
    return {
        "text": (
            f"### Instruction:\n{ex['instruction']}\n\n"
            f"### Input:\n{ex['input']}\n\n"
            f"### Response:\n{ex['output']}"
        )
    }

train_data = Dataset.from_list(qa_dataset).map(format_train)

tokenized_data = train_data.map(
    lambda x: tokenizer(
        x["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    ),
    batched=True,
    remove_columns=train_data.column_names # Bu satırı ekleyerek orijinal string sütunları kaldırıyoruz
)

print(tokenized_data[0])

Map:   0%|          | 0/74 [00:00<?, ? examples/s]

Map:   0%|          | 0/74 [00:00<?, ? examples/s]

{'input_ids': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 835, 2799, 4080, 29901, 13, 22550, 408, 385, 1095, 397, 609, 391, 773, 278, 2183, 3030, 29889, 13, 13, 2277, 29937, 10567, 29901, 13, 6007, 16975, 29901, 13, 29889, 1152, 1342, 29892, 341, 6040, 2514, 433, 412, 29916, 75

In [ ]:
args = TrainingArguments(
    output_dir="./endodonti_fine_tuned_model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_data,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.663632
20,1.328236
30,1.231066
40,1.216398


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=40, training_loss=1.3598331451416015, metrics={'train_runtime': 118.3626, 'train_samples_per_second': 2.501, 'train_steps_per_second': 0.338, 'total_flos': 944791521067008.0, 'train_loss': 1.3598331451416015, 'epoch': 4.0})

In [ ]:
trainer.model.save_pretrained("./endodonti_fine_tuned_model")
tokenizer.save_pretrained("./endodonti_fine_tuned_model")
print("✅ Eğitim bitti! adapter_config.json ve model ağırlıkları hazır.")

✅ Eğitim bitti! adapter_config.json ve model ağırlıkları hazır.


In [ ]:
def asistan_test_et(soru, context):
    model.eval()
    test_prompt = (
        f"### Instruction:\nAnswer as an endodontist using the given context.\n\n"
        f"### Input:\nCONTEXT:\n{context}\n\nQuestion: {soru}\n\n"
        f"### Response:\n"
    )

    inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.4,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True).split("### Response:\n")[-1]

    print("\n" + "="*50)
    print(f"🤔 SORU: {soru}")
    print(f"👨‍⚕️ ASİSTANIN CEVABI:\n{answer.strip()}")
    print("="*50)

In [ ]:
ornek_context = rows[0]["text"][:1000]

asistan_test_et(
    "Sodyum hipokloritin (NaOCl) kök kanal tedavisindeki temel amaçları nelerdir?",
    ornek_context
)

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🤔 SORU: Sodyum hipokloritin (NaOCl) kök kanal tedavisindeki temel amaçları nelerdir?
👨‍⚕️ ASİSTANIN CEVABI:
Sodium hypochlorite (NaOCl) is used for closing canals in endodontology to prevent bacterial growth and promote root canal filling. Its primary purpose is to kill germs that could cause infection during treatment.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('endodonti_final_paket', 'zip', './endodonti_fine_tuned_model')
files.download('endodonti_final_paket.zip')
print("🚀 Paket indiriliyor. İçinde adapter_config.json var mı kontrol et!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🚀 Paket indiriliyor. İçinde adapter_config.json var mı kontrol et!
